In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import random
import pandas as pd

# Add the parent directory to Python's module search path
sys.path.append(os.path.abspath('..'))
from src.bpe import BPETokenizer

# 1. Define Noise Injection Functions
def inject_typographic_noise(text, error_rate=0.1):
    """Simulates fast typing errors (character swaps)."""
    chars = list(text)
    for i in range(len(chars) - 1):
        if random.random() < error_rate:
            # Swap adjacent characters
            chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def simulate_ocr_error(text, error_rate=0.05):
    """Simulates visual misreadings by OCR engines."""
    replacements = {'o': '0', 'l': '1', 's': '5', 'i': '!', 'a': '@', 'e': 'c'}
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < error_rate and chars[i].lower() in replacements:
            chars[i] = replacements[chars[i].lower()]
    return "".join(chars)

# 2. Load the trained 8K Tokenizer
tokenizer = BPETokenizer()
vocab_file = '../vocabs/bpe_8000.json'

if os.path.exists(vocab_file):
    print(" Loading 8K vocabulary for robustness testing...")
    tokenizer.load(vocab_file)
else:
    print(" Error: 8K vocab not found. Please ensure it trained successfully.")

# 3. Create a Test Dataset
test_sentences = [
    "Natural language processing requires highly efficient subword tokenization.",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models are heavily dependent on the quality of their training data.",
    "Standardization of subword units is essential for multilingual systems."
]

results = []

print("\n--- Running Robustness Evaluation ---")
for sentence in test_sentences:
    # Generate Noisy Variants
    noisy_typo = inject_typographic_noise(sentence, error_rate=0.1)
    noisy_ocr = simulate_ocr_error(sentence, error_rate=0.1)
    
    # Tokenize all variants
    clean_eval = tokenizer.encode(sentence)
    typo_eval = tokenizer.encode(noisy_typo)
    ocr_eval = tokenizer.encode(noisy_ocr)
    
    # Calculate Fragmentation (Increase in tokens compared to clean text)
    clean_len = len(clean_eval['tokens'])
    typo_len = len(typo_eval['tokens'])
    ocr_len = len(ocr_eval['tokens'])
    
    results.append({
        "Condition": "Clean",
        "Text Sample": sentence[:40] + "...",
        "Token Count": clean_len,
        "OOV Rate": round(clean_eval['oov_rate'], 4),
        "Tokens/Word": round(clean_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "Typo Noise",
        "Text Sample": noisy_typo[:40] + "...",
        "Token Count": typo_len,
        "OOV Rate": round(typo_eval['oov_rate'], 4),
        "Tokens/Word": round(typo_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "OCR Noise",
        "Text Sample": noisy_ocr[:40] + "...",
        "Token Count": ocr_len,
        "OOV Rate": round(ocr_eval['oov_rate'], 4),
        "Tokens/Word": round(ocr_eval['tokens_per_word'], 2)
    })

# 4. Display the Results
df_robustness = pd.DataFrame(results)

# Calculate aggregate degradation
clean_avg_tokens = df_robustness[df_robustness['Condition'] == 'Clean']['Token Count'].mean()
typo_avg_tokens = df_robustness[df_robustness['Condition'] == 'Typo Noise']['Token Count'].mean()
ocr_avg_tokens = df_robustness[df_robustness['Condition'] == 'OCR Noise']['Token Count'].mean()

print(f"\n Average Tokenization Length:")
print(f"  Clean Text: {clean_avg_tokens:.1f} tokens")
print(f"  Typo Noise: {typo_avg_tokens:.1f} tokens (+{((typo_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")
print(f"  OCR Noise:  {ocr_avg_tokens:.1f} tokens (+{((ocr_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")

display(df_robustness)

# Save for the report
os.makedirs('../reports/', exist_ok=True)
df_robustness.to_csv('../reports/robustness_metrics.csv', index=False)

 Loading 8K vocabulary for robustness testing...

--- Running Robustness Evaluation ---

 Average Tokenization Length:
  Clean Text: 16.8 tokens
  Typo Noise: 25.8 tokens (+53.7%)
  OCR Noise:  22.2 tokens (+32.8%)


,Condition,Text Sample,Token Count,OOV Rate,Tokens/Word
0,Clean,Natural language processing requires hig...,18,0.0,2.00
1,Typo Noise,aNturla language prcoessing requires hig...,26,0.0,2.89
2,OCR Noise,Natural 1angu@ge pr0cessing rcquires hig...,30,0.0,2.31
3,Clean,The quick brown fox jumps over the lazy ...,16,0.0,1.60
4,Typo Noise,The quick brownf o xjump svoer the lazy ...,21,0.0,2.10
5,OCR Noise,The quick brown fox jumps over the lazy ...,16,0.0,1.60
6,Clean,Machine learning models are heavily depe...,17,0.0,1.21
7,Typo Noise,Mcaihen leanring omdels are heavilyd epe...,31,0.0,2.21
8,OCR Noise,Machine 1earn!ng models are heavily depe...,24,0.0,1.50
9,Clean,Standardization of subword units is esse...,16,0.0,1.60


In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import random
import pandas as pd

# Add the parent directory to Python's module search path
sys.path.append(os.path.abspath('..'))
from src.bpe import BPETokenizer

# 1. Define Noise Injection Functions
def inject_typographic_noise(text, error_rate=0.1):
    """Simulates fast typing errors (character swaps)."""
    chars = list(text)
    for i in range(len(chars) - 1):
        if random.random() < error_rate:
            # Swap adjacent characters
            chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def simulate_ocr_error(text, error_rate=0.05):
    """Simulates visual misreadings by OCR engines."""
    replacements = {'o': '0', 'l': '1', 's': '5', 'i': '!', 'a': '@', 'e': 'c'}
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < error_rate and chars[i].lower() in replacements:
            chars[i] = replacements[chars[i].lower()]
    return "".join(chars)

# 2. Load the trained 8K Tokenizer
tokenizer = BPETokenizer()
vocab_file = '../vocabs/bpe_16000.json'

if os.path.exists(vocab_file):
    print(" Loading 16K vocabulary for robustness testing...")
    tokenizer.load(vocab_file)
else:
    print(" Error: 16K vocab not found. Please ensure it trained successfully.")

# 3. Create a Test Dataset
test_sentences = [
    "Natural language processing requires highly efficient subword tokenization.",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models are heavily dependent on the quality of their training data.",
    "Standardization of subword units is essential for multilingual systems."
]

results = []

print("\n--- Running Robustness Evaluation ---")
for sentence in test_sentences:
    # Generate Noisy Variants
    noisy_typo = inject_typographic_noise(sentence, error_rate=0.1)
    noisy_ocr = simulate_ocr_error(sentence, error_rate=0.1)
    
    # Tokenize all variants
    clean_eval = tokenizer.encode(sentence)
    typo_eval = tokenizer.encode(noisy_typo)
    ocr_eval = tokenizer.encode(noisy_ocr)
    
    # Calculate Fragmentation (Increase in tokens compared to clean text)
    clean_len = len(clean_eval['tokens'])
    typo_len = len(typo_eval['tokens'])
    ocr_len = len(ocr_eval['tokens'])
    
    results.append({
        "Condition": "Clean",
        "Text Sample": sentence[:40] + "...",
        "Token Count": clean_len,
        "OOV Rate": round(clean_eval['oov_rate'], 4),
        "Tokens/Word": round(clean_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "Typo Noise",
        "Text Sample": noisy_typo[:40] + "...",
        "Token Count": typo_len,
        "OOV Rate": round(typo_eval['oov_rate'], 4),
        "Tokens/Word": round(typo_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "OCR Noise",
        "Text Sample": noisy_ocr[:40] + "...",
        "Token Count": ocr_len,
        "OOV Rate": round(ocr_eval['oov_rate'], 4),
        "Tokens/Word": round(ocr_eval['tokens_per_word'], 2)
    })

# 4. Display the Results
df_robustness = pd.DataFrame(results)

# Calculate aggregate degradation
clean_avg_tokens = df_robustness[df_robustness['Condition'] == 'Clean']['Token Count'].mean()
typo_avg_tokens = df_robustness[df_robustness['Condition'] == 'Typo Noise']['Token Count'].mean()
ocr_avg_tokens = df_robustness[df_robustness['Condition'] == 'OCR Noise']['Token Count'].mean()

print(f"\n Average Tokenization Length:")
print(f"  Clean Text: {clean_avg_tokens:.1f} tokens")
print(f"  Typo Noise: {typo_avg_tokens:.1f} tokens (+{((typo_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")
print(f"  OCR Noise:  {ocr_avg_tokens:.1f} tokens (+{((ocr_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")

display(df_robustness)

# Save for the report
os.makedirs('../reports/', exist_ok=True)
df_robustness.to_csv('../reports/robustness_metrics.csv', index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
 Loading 16K vocabulary for robustness testing...

--- Running Robustness Evaluation ---

 Average Tokenization Length:
  Clean Text: 14.5 tokens
  Typo Noise: 21.0 tokens (+44.8%)
  OCR Noise:  21.5 tokens (+48.3%)


,Condition,Text Sample,Token Count,OOV Rate,Tokens/Word
0,Clean,Natural language processing requires hig...,15,0.0,1.67
1,Typo Noise,aNtural language pocressnig reqiurse hgi...,32,0.0,3.56
2,OCR Noise,Natur@l language proccssing requires hig...,23,0.0,1.35
3,Clean,The quick brown fox jumps over the lazy ...,13,0.0,1.30
4,Typo Noise,The quick brown fox jumps over the lazy ...,13,0.0,1.30
5,OCR Noise,The quick brown fox jumps over the lazy ...,13,0.0,1.30
6,Clean,Machine learning models are heavily depe...,14,0.0,1.00
7,Typo Noise,Machine learning models are heavily depe...,18,0.0,1.29
8,OCR Noise,Mach!ne learning mode1s are heavily depe...,26,0.0,1.24
9,Clean,Standardization of subword units is esse...,16,0.0,1.60


In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
import random
import pandas as pd

# Add the parent directory to Python's module search path
sys.path.append(os.path.abspath('..'))
from src.bpe import BPETokenizer

# 1. Define Noise Injection Functions
def inject_typographic_noise(text, error_rate=0.1):
    """Simulates fast typing errors (character swaps)."""
    chars = list(text)
    for i in range(len(chars) - 1):
        if random.random() < error_rate:
            # Swap adjacent characters
            chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def simulate_ocr_error(text, error_rate=0.05):
    """Simulates visual misreadings by OCR engines."""
    replacements = {'o': '0', 'l': '1', 's': '5', 'i': '!', 'a': '@', 'e': 'c'}
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < error_rate and chars[i].lower() in replacements:
            chars[i] = replacements[chars[i].lower()]
    return "".join(chars)

# 2. Load the trained 8K Tokenizer
tokenizer = BPETokenizer()
vocab_file = '../vocabs/bpe_32000.json'

if os.path.exists(vocab_file):
    print(" Loading 32K vocabulary for robustness testing...")
    tokenizer.load(vocab_file)
else:
    print(" Error: 32K vocab not found. Please ensure it trained successfully.")

# 3. Create a Test Dataset
test_sentences = [
    "Natural language processing requires highly efficient subword tokenization.",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models are heavily dependent on the quality of their training data.",
    "Standardization of subword units is essential for multilingual systems."
]

results = []

print("\n--- Running Robustness Evaluation ---")
for sentence in test_sentences:
    # Generate Noisy Variants
    noisy_typo = inject_typographic_noise(sentence, error_rate=0.1)
    noisy_ocr = simulate_ocr_error(sentence, error_rate=0.1)
    
    # Tokenize all variants
    clean_eval = tokenizer.encode(sentence)
    typo_eval = tokenizer.encode(noisy_typo)
    ocr_eval = tokenizer.encode(noisy_ocr)
    
    # Calculate Fragmentation (Increase in tokens compared to clean text)
    clean_len = len(clean_eval['tokens'])
    typo_len = len(typo_eval['tokens'])
    ocr_len = len(ocr_eval['tokens'])
    
    results.append({
        "Condition": "Clean",
        "Text Sample": sentence[:40] + "...",
        "Token Count": clean_len,
        "OOV Rate": round(clean_eval['oov_rate'], 4),
        "Tokens/Word": round(clean_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "Typo Noise",
        "Text Sample": noisy_typo[:40] + "...",
        "Token Count": typo_len,
        "OOV Rate": round(typo_eval['oov_rate'], 4),
        "Tokens/Word": round(typo_eval['tokens_per_word'], 2)
    })
    
    results.append({
        "Condition": "OCR Noise",
        "Text Sample": noisy_ocr[:40] + "...",
        "Token Count": ocr_len,
        "OOV Rate": round(ocr_eval['oov_rate'], 4),
        "Tokens/Word": round(ocr_eval['tokens_per_word'], 2)
    })

# 4. Display the Results
df_robustness = pd.DataFrame(results)

# Calculate aggregate degradation
clean_avg_tokens = df_robustness[df_robustness['Condition'] == 'Clean']['Token Count'].mean()
typo_avg_tokens = df_robustness[df_robustness['Condition'] == 'Typo Noise']['Token Count'].mean()
ocr_avg_tokens = df_robustness[df_robustness['Condition'] == 'OCR Noise']['Token Count'].mean()

print(f"\n Average Tokenization Length:")
print(f"  Clean Text: {clean_avg_tokens:.1f} tokens")
print(f"  Typo Noise: {typo_avg_tokens:.1f} tokens (+{((typo_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")
print(f"  OCR Noise:  {ocr_avg_tokens:.1f} tokens (+{((ocr_avg_tokens/clean_avg_tokens)-1)*100:.1f}%)")

display(df_robustness)

# Save for the report
os.makedirs('../reports/', exist_ok=True)
df_robustness.to_csv('../reports/robustness_metrics.csv', index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
 Loading 32K vocabulary for robustness testing...

--- Running Robustness Evaluation ---

 Average Tokenization Length:
  Clean Text: 13.2 tokens
  Typo Noise: 21.8 tokens (+64.2%)
  OCR Noise:  20.2 tokens (+52.8%)


,Condition,Text Sample,Token Count,OOV Rate,Tokens/Word
0,Clean,Natural language processing requires hig...,12,0.0,1.33
1,Typo Noise,Natural language procsseing requires hig...,23,0.0,2.56
2,OCR Noise,Natural language processing rcquires h!g...,18,0.0,1.64
3,Clean,The quick brown fox jumps over the lazy ...,11,0.0,1.10
4,Typo Noise,The quickb rown fox jumps over the lazy ...,14,0.0,1.40
5,OCR Noise,The quick brown fox jumps 0ver the 1azy ...,13,0.0,1.30
6,Clean,Machine learning models are heavily depe...,14,0.0,1.00
7,Typo Noise,Machine learinng models are hevaiyl depe...,24,0.0,1.71
8,OCR Noise,Mach!ne learning mode1s are heavily depe...,25,0.0,1.47
9,Clean,Standardization of subword units is esse...,16,0.0,1.60
